# Beyond Nyquist — 실수 신호의 대칭성을 이용한 절반 레이트 샘플링

## 문제 의식

샘플링 정리는 대역폭 $B$인 신호에 대해 $\omega_s > 2B$를 요구한다. 이 한계가 ADC 속도, 비용, 전력에 직결되는 공학적 제약이다.

질문: **이 한계를 깨고 절반의 레이트로 정보를 보존할 수 있는가?**

이 노트북은 답을 단계별로 쌓는다. 가우시안 스펙트럼이라는 *우함수 실수* 신호의 단순한 경우에서 출발해 — 첫 번째 막힘(복소 신호 문제) — 실수 신호가 만족해야 할 대칭 조건의 재발견 — **\"aliasing을 aliasing으로 상쇄한다\"** 는 아이디어 — Hilbert transform 의 자연스러운 등장 — 비인과성과 회로 구현(all-pass cascade) — polyphase 우회까지 한 흐름으로 이어진다. 이 결과는 SDR, SSB 송수신기, MRI 등에서 **IQ sampling / phasing method** 라는 이름으로 광범위하게 사용된다.

---
[GitHub repository](https://github.com/dohyunKim0309/Fourier-Transform-Visualization) · [Open in Colab](https://colab.research.google.com/github/dohyunKim0309/Fourier-Transform-Visualization/blob/main/examples/05_beyond_nyquist_hilbert.ipynb)

*2026년 5월 · 연세대학교 전기전자공학부 22학번 김도현*


In [1]:
# Colab/Jupyter에서 ftvis 라이브러리 설치 (이미 설치돼 있으면 즉시 통과)
%pip install -q git+https://github.com/dohyunKim0309/Fourier-Transform-Visualization.git


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: /Users/gimdohyeon/PycharmProjects/Signals_Systems/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from ftvis import SpectrumPlot

# ─────────────────────────────────────────────────────────────────────────
# 다른 노트북과 동일 테마 (ftvis/viz.py와 일치)
# ─────────────────────────────────────────────────────────────────────────
_BG       = '#0d1117'
_FG       = '#e6edf3'
_GRID     = '#30363d'
_AXIS     = '#7d8590'
_GREEN    = '#7ec699'   # signal·spectrum 본체
_BLUE     = '#58a6ff'   # winding / 보조
_ORANGE   = '#ffa657'   # 강조·alias component
_RED      = '#ff7b72'   # 부호 반대 강조
_PURPLE   = '#bf91f3'   # imag part
_YELLOW   = '#e3b341'   # 합
_WHITE    = '#f0f6fc'

_DEFAULT_LAYOUT = dict(
    paper_bgcolor=_BG, plot_bgcolor=_BG,
    font=dict(color=_FG, family='serif', size=12),
    margin=dict(l=10, r=10, t=50, b=10),
)
_2D_AXIS = dict(showgrid=True, gridcolor=_GRID, zeroline=True,
                zerolinecolor=_AXIS, linecolor=_AXIS, color=_FG)
_3D_AXIS = dict(backgroundcolor=_BG, gridcolor=_GRID,
                zerolinecolor=_AXIS, showbackground=True, color=_FG)


def fig2d(title='', height=320, xtitle='ω', ytitle='X(jω)'):
    fig = go.Figure()
    fig.update_layout(
        **_DEFAULT_LAYOUT,
        title=dict(text=title, x=0.02, xanchor='left', font=dict(color=_FG)),
        height=height,
        xaxis=dict(_2D_AXIS, title=xtitle),
        yaxis=dict(_2D_AXIS, title=ytitle),
        showlegend=True,
        legend=dict(bgcolor='rgba(0,0,0,0)', font=dict(color=_FG)),
    )
    return fig


def fig3d(title='', height=520, xtitle='ω', ytitle='Re', ztitle='Im',
          aspect=(2.4, 1.0, 1.0), eye=(2.2, 1.5, 0.9)):
    fig = go.Figure()
    fig.update_layout(
        **_DEFAULT_LAYOUT,
        title=dict(text=title, x=0.02, xanchor='left', font=dict(color=_FG)),
        height=height,
        scene=dict(
            xaxis=dict(_3D_AXIS, title=xtitle),
            yaxis=dict(_3D_AXIS, title=ytitle),
            zaxis=dict(_3D_AXIS, title=ztitle),
            aspectmode='manual',
            aspectratio=dict(x=aspect[0], y=aspect[1], z=aspect[2]),
            camera=dict(eye=dict(x=eye[0], y=eye[1], z=eye[2])),
        ),
        showlegend=True,
        legend=dict(bgcolor='rgba(0,0,0,0)', font=dict(color=_FG)),
    )
    return fig


def add_central_axis(fig, u_range, *, color=_AXIS, width=2, scene='scene'):
    '''3D 씬에 (u, 0, 0) 중심축을 한 줄 그어준다.'''
    u0, u1 = float(u_range[0]), float(u_range[1])
    fig.add_trace(go.Scatter3d(
        x=[u0, u1], y=[0, 0], z=[0, 0],
        mode='lines', line=dict(color=color, width=width),
        showlegend=False, hoverinfo='skip',
    ))


def add_plane(fig, *, axis='y', value=0.0, u_range=(-1, 1), w_range=(-1, 1),
              color='rgba(126,198,153,0.10)', name=''):
    '''3D 씬에 반투명 평면을 추가. axis는 평면이 *상수인* 축.
    axis='z' value=0 → Im=0 평면 (실수-시간/ω 평면).
    axis='y' value=0 → Re=0 평면.
    '''
    u0, u1 = u_range; w0, w1 = w_range
    if axis == 'z':
        x = [u0, u1, u1, u0]; y = [w0, w0, w1, w1]; z = [value]*4
    elif axis == 'y':
        x = [u0, u1, u1, u0]; z = [w0, w0, w1, w1]; y = [value]*4
    else:  # axis == 'x'
        y = [u0, u1, u1, u0]; z = [w0, w0, w1, w1]; x = [value]*4
    fig.add_trace(go.Mesh3d(
        x=x, y=y, z=z, i=[0, 0], j=[1, 2], k=[2, 3],
        color=color, opacity=1.0, showlegend=bool(name), name=name,
        hoverinfo='skip', flatshading=True,
    ))


print('imports + theme ready')

imports + theme ready


## [1] 첫 번째 아이디어: 우함수 스펙트럼의 대칭성

가장 단순한 경우부터. **$X(j\omega)$가 순실수이고 우함수**인 신호 — 시간 영역에서도 우함수 실수 신호다.

가우시안 $x(t) = e^{-t^2/(2\sigma^2)}$의 스펙트럼은

$$X(j\omega) = \sigma\sqrt{2\pi}\,e^{-\sigma^2\omega^2/2}$$

대역폭을 $B$로 잘라 보면 (truncated Gaussian) $X(j\omega) = X(-j\omega)$. **양/음 주파수가 같은 값.** 양의 주파수만 알면 음의 주파수도 안다 — 정보가 사실상 절반 폭에만 있다.

> *직관*: 음의 주파수 부분을 "날려도" 정보 손실 없다. 폭이 절반이니 샘플레이트도 절반이면 충분하지 않을까?

In [3]:
# 신호: -B에서 +B까지 가우시안, 나머지는 0 (truncated). 우함수.
# 음의 절반을 흐리게 그려도 정보는 양의 절반에 다 있다는 것을 보여준다.
sigma = 0.6
B = 3.0                                       # 대역폭 (rad/s) - support [-B, +B]
omega = np.linspace(-1.6*B, 1.6*B, 1201)      # 잘림이 시각적으로 보이게 양옆 여유

# truncated Gaussian: |ω| > B 인 곳은 정확히 0
X = np.where(np.abs(omega) <= B,
             sigma * np.sqrt(2*np.pi) * np.exp(-0.5 * (sigma * omega)**2),
             0.0)

pos_mask = (omega >= 0) & (omega <= B)
neg_mask = (omega < 0)  & (omega >= -B)

fig = fig2d(title=f'[1] Truncated Gaussian  X(jω) = X(−jω),  support [−B, +B] (B={B})  '
                  '— 양의 절반에 정보 전부',
            height=340, ytitle='X(jω)  (real, even)')
# 음의 절반: 흐리게
fig.add_trace(go.Scatter(
    x=omega[neg_mask], y=X[neg_mask], mode='lines',
    line=dict(color=_GREEN, width=2, dash='dot'), opacity=0.45,
    name='ω ∈ [−B, 0]  (음의 절반: 양의 절반으로 복원 가능)',
))
# 양의 절반: 진하게
fig.add_trace(go.Scatter(
    x=omega[pos_mask], y=X[pos_mask], mode='lines',
    line=dict(color=_ORANGE, width=4),
    name='ω ∈ [0, +B]  (정보 전체를 담은 절반)',
))
# truncation 경계
fig.add_vline(x=+B, line=dict(color=_RED, width=1, dash='dash'),
              annotation_text='+B', annotation=dict(font=dict(color=_RED)))
fig.add_vline(x=-B, line=dict(color=_RED, width=1, dash='dash'),
              annotation_text='−B', annotation=dict(font=dict(color=_RED)))
fig.add_vline(x=0, line=dict(color=_AXIS, width=1, dash='dash'))
fig.show()

## [2] 첫 번째 장벽: 한쪽 sideband를 날리면 *복소 신호*

음의 주파수만 떼낸 *한쪽 sideband* 스펙트럼

$$X_+(j\omega) = X(j\omega)\cdot u(\omega), \qquad u: \text{단위 계단}$$

이걸 시간 도메인으로 역변환하면 어떻게 될까? 우함수 대칭이 깨졌으니 결과는 **시간 도메인에서 복소 신호** 가 된다 (왜 그런지는 §4 에서 정리).

3D로 직접 보자. 시간축 $t$ + 복소평면 $(\mathrm{Re}, \mathrm{Im})$. 만약 $x_+(t)$가 실수였다면 *전체 곡선이 $\mathrm{Im} = 0$ 평면 위에 누워야* 한다 (배경의 반투명 청록색 평면). 실제로는 그 평면을 벗어나 헬릭스처럼 휘어 올라간다 — 한쪽 sideband를 남긴 결과는 *analytic signal* 로 알려진 복소 신호다.

> **함의**: 아날로그 도메인의 모든 물리량(전압, 전류, 전기장)은 실수. 따라서 한쪽 sideband만 가진 신호는 **물리적으로 측정 불가능, ADC가 받을 수 없다.** 아이디어 자체는 옳지만 직접 실현 경로가 막힌다.

In [4]:
# truncated Gaussian의 X_+(jω) = X(jω)·u(ω) 역변환 — 시간 도메인에서 복소 신호.

# X는 [-B, +B] support, 그 중 양의 절반 [0, +B]만 살림.

sigma = 0.6
B = 3.0                       # 신호 대역폭
n_omega = 2001
omega = np.linspace(-B, B, n_omega)      # truncate된 신호이므로 [-B, B]만 적분
dw = omega[1] - omega[0]
X = sigma * np.sqrt(2*np.pi) * np.exp(-0.5 * (sigma*omega)**2)
X_plus = X.copy()
X_plus[omega < 0] = 0.0                                 # 한쪽 sideband (ω ∈ [0, +B])

# 시간 격자: 길게 펴서 헬릭스가 잘 보이게
t = np.linspace(-6.0, 6.0, 481)
# x_+(t) = (1/2π) ∫ X_+(ω) e^(jωt) dω  ≈ (1/2π) Σ X_+(ω_k) e^(jω_k t) dω
x_plus = (X_plus[None, :] * np.exp(1j * np.outer(t, omega))).sum(axis=1) * dw / (2*np.pi)

# 비교용: 원본 truncated Gaussian (실수)의 IFT
x_orig = (X[None, :] * np.exp(1j * np.outer(t, omega))).sum(axis=1).real * dw / (2*np.pi)

re_amp = max(float(np.max(np.abs(x_plus.real))), float(np.max(np.abs(x_orig)))) * 1.2
fig = fig3d(
    title='[2] 한쪽 sideband의 역변환 — 실수가 아닌 복소 헬릭스 (Im=0 평면을 벗어남)',
    height=560, xtitle='t', ytitle='Re x₊(t)', ztitle='Im x₊(t)',
    aspect=(2.6, 1.0, 1.0), eye=(2.1, 1.6, 0.85),
)
add_central_axis(fig, (t[0], t[-1]))
# Im=0 평면 (실수 신호가 차지해야 할 자리)
add_plane(fig, axis='z', value=0.0,
          u_range=(t[0], t[-1]), w_range=(-re_amp, re_amp),
          color='rgba(126,198,153,0.10)', name='Im = 0 평면 (실수 신호 자리)')
# 원본 실수 truncated Gaussian의 IFT (Im=0 평면 위)
fig.add_trace(go.Scatter3d(
    x=t, y=x_orig, z=np.zeros_like(t), mode='lines',
    line=dict(color=_GREEN, width=3, dash='dot'),
    name='원본 실수 신호 x(t) (참고)',
))
# x_+(t): 3D 헬릭스
fig.add_trace(go.Scatter3d(
    x=t, y=x_plus.real, z=x_plus.imag, mode='lines',
    line=dict(color=_ORANGE, width=5),
    name='x₊(t) = IFT{X·u(ω)}  — 복소 헬릭스',
))
fig.show()

ratio_im_re = float(np.max(np.abs(x_plus.imag))) / float(np.max(np.abs(x_plus.real)))
print(f'max|Im x_+| / max|Re x_+| = {ratio_im_re:.4f}   '
      f'(실수 신호면 0이어야 함 → 0이 아니라 복소)')

max|Im x_+| / max|Re x_+| = 0.6548   (실수 신호면 0이어야 함 → 0이 아니라 복소)


## [3] 우회 시도: LPF/HPF 분할 + Mixing

신호를 LPF/HPF로 두 절반으로 나누고, HPF 결과에 $\cos(\omega_c t)$를 곱해 (mixing) baseband로 내린 뒤 따로 샘플링, 디지털 재결합. 분명히 가능하지만:

- 두 개의 분리된 ADC 체인 + mixer + 매칭된 부품 → **단계가 많음**
- 원래의 깔끔한 대칭성 아이디어를 *직접 살리지 못함*. 정보가 절반에 있다는 사실을 시스템 구조가 반영하지 못하고 우회로 푼다.

더 직접적인 길이 없을까? — 일단 LPF/HPF가 ω 도메인에서 어떻게 보이는지부터.

In [5]:
# 우함수 가우시안 스펙트럼을 LPF / HPF로 분할 (양의 절반만 표시)
sigma = 1.0
B = 4.0
omega_c = 1.6                          # 분할 컷오프
omega = np.linspace(-B, B, 1201)
X = sigma * np.sqrt(2*np.pi) * np.exp(-0.5 * (sigma*omega)**2)

# 부드러운 LPF/HPF (브릭월 대신 cos² 천이 — 시각이 매끄러움)
transition = 0.25
def smooth_window(w, wc, kind='lpf'):
    a = wc - transition; b = wc + transition
    aw = np.abs(w)
    lp = np.where(aw <= a, 1.0,
         np.where(aw >= b, 0.0,
                  0.5*(1 + np.cos(np.pi*(aw - a)/(b - a)))))
    return lp if kind == 'lpf' else 1.0 - lp

H_lp = smooth_window(omega, omega_c, 'lpf')
H_hp = smooth_window(omega, omega_c, 'hpf')
X_lp = X * H_lp
X_hp = X * H_hp

# Mixing: HPF 결과에 cos(ω_c t) 곱 → 주파수 도메인에서 ±ω_c 만큼 시프트 (양분)
X_mixed = 0.5 * (np.interp(omega - omega_c, omega, X_hp)
                + np.interp(omega + omega_c, omega, X_hp))

fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.08,
                    subplot_titles=('LPF 통과: X·H_lp(ω)',
                                    'HPF 통과: X·H_hp(ω)',
                                    'HPF · cos(ω_c t) → mixing'))
for col, (Y, color, label) in enumerate([
        (X_lp, _GREEN, 'X·H_lp'),
        (X_hp, _ORANGE, 'X·H_hp'),
        (X_mixed, _BLUE, '½[X_hp(ω−ω_c)+X_hp(ω+ω_c)]')], start=1):
    fig.add_trace(go.Scatter(x=omega, y=X, mode='lines',
                              line=dict(color=_AXIS, width=1, dash='dot'),
                              name='X(jω)' if col == 1 else None,
                              showlegend=(col == 1)),
                  row=1, col=col)
    fig.add_trace(go.Scatter(x=omega, y=Y, mode='lines',
                              line=dict(color=color, width=3),
                              fill='tozeroy',
                              fillcolor=color.replace(')', ', 0.25)').replace('#', 'rgba(')
                                       if False else None,
                              name=label, showlegend=False),
                  row=1, col=col)
    fig.add_vline(x=omega_c, line=dict(color=_AXIS, width=1, dash='dash'),
                  row=1, col=col)
    fig.add_vline(x=-omega_c, line=dict(color=_AXIS, width=1, dash='dash'),
                  row=1, col=col)
fig.update_layout(
    **_DEFAULT_LAYOUT, height=320,
    title=dict(text=f'[3] LPF/HPF 분할 + mixing (ω_c = {omega_c})  '
                    '— 두 ADC 체인이 필요한 우회 방식',
               x=0.02, xanchor='left', font=dict(color=_FG)),
    showlegend=False,
)
for c in (1, 2, 3):
    fig.update_xaxes(_2D_AXIS, title='ω', row=1, col=c)
    fig.update_yaxes(_2D_AXIS, row=1, col=c)
for ann in fig['layout']['annotations']:
    ann['font'] = dict(color=_FG, size=12)
fig.show()

## [4] 가정 재검토: $x(t)$가 실수일 필요충분조건

첫 아이디어의 진짜 가정이 뭐였는지 다시 본다. "$X(j\omega)$가 실수이고 우함수"는 *너무 강한* 가정 — $\cos$ 계열 같은 **실수 우함수 신호** 만 다룬다는 뜻이다. 일반 실수 신호는 어떤 모양인가?

$x(t)$가 실수일 필요충분조건은 $X(j\omega)$의 **Hermitian 대칭**:

$$X(-j\omega) = \overline{X(j\omega)}$$

즉:

- 실수부 $X_R(\omega)$ — **우함수**
- 허수부 $X_I(\omega)$ — **기함수**

처음의 가정은 $X_I = 0$인 *특수 경우*였다. 일반 실수 신호는 Hermitian 대칭만 만족하면 되고, 허수부가 0일 필요는 없다.

기하적으로 보자: 복소수 $X(j\omega)$는 $\omega$-Re-Im 3D 공간의 점. Re는 $\omega$ 우함수 (Re 평면에 가우시안), Im은 $\omega$ 기함수 (Im 평면에 ω·exp 같은 곡선). 두 곡선이 한 3D 공간에서 어떻게 살아 있는지 본다.

In [6]:
# Re-ω 평면의 우함수 (가우시안) + Im-ω 평면의 기함수 (정규화 derivative-of-Gaussian)
# 그리고 두 부분을 합친 일반 실수 신호의 X(jω)를 3D ω-Re-Im에 동시 표시.

sigma = 1.0
B = 4.0
omega = np.linspace(-B, B, 1001)

# Re: 가우시안 (우함수)
X_R = sigma * np.sqrt(2*np.pi) * np.exp(-0.5 * (sigma*omega)**2)
# Im: derivative-of-Gaussian 계열 — ω · exp(-σ²ω²/2) (기함수)
X_I = 1.4 * omega * np.exp(-0.5 * (sigma*omega)**2)

X = X_R + 1j * X_I  # 일반 실수 신호 x(t)의 스펙트럼

# (a) 2D — Re-ω 평면 우함수 + Im-ω 평면 기함수
fig_2d = make_subplots(rows=1, cols=2, horizontal_spacing=0.10,
                       subplot_titles=('Re X(jω) — 우함수 (예: 가우시안)',
                                       'Im X(jω) — 기함수 (예: ω·e^(−σ²ω²/2))'))
fig_2d.add_trace(go.Scatter(x=omega, y=X_R, mode='lines',
                            line=dict(color=_GREEN, width=3), fill='tozeroy',
                            fillcolor='rgba(126,198,153,0.20)', showlegend=False),
                 row=1, col=1)
fig_2d.add_trace(go.Scatter(x=omega, y=X_I, mode='lines',
                            line=dict(color=_PURPLE, width=3), fill='tozeroy',
                            fillcolor='rgba(191,145,243,0.20)', showlegend=False),
                 row=1, col=2)
for c in (1, 2):
    fig_2d.add_vline(x=0, line=dict(color=_AXIS, width=1, dash='dash'), row=1, col=c)
    fig_2d.update_xaxes(_2D_AXIS, title='ω', row=1, col=c)
    fig_2d.update_yaxes(_2D_AXIS, row=1, col=c)
fig_2d.update_layout(
    **_DEFAULT_LAYOUT, height=300,
    title=dict(text='[4a] Hermitian 대칭의 두 성분 — Re는 우함수, Im은 기함수',
               x=0.02, xanchor='left', font=dict(color=_FG)),
    showlegend=False,
)
for ann in fig_2d['layout']['annotations']:
    ann['font'] = dict(color=_FG, size=12)
fig_2d.show()

# (b) 3D — 일반 실수 신호의 X(jω)를 ω-Re-Im에 (SpectrumPlot 사용)
panel = SpectrumPlot(view='re_im', show_inset=True)
panel.show_spectrum(omega, X, flatten_noise=False)
panel.figure.layout.title.text = (
    '[4b] 일반 실수 신호의 X(jω) — Re 우함수 + Im 기함수가 한 곡선으로'
)
panel.figure.show()

## [5] Aliasing 을 Aliasing 으로 상쇄한다

이 절의 아이디어 하나가 노트북 전체를 끌어간다.

신호 $X(j\omega)$ 는 $[-B, +B]$ 에 truncated된 가우시안. **Nyquist rate 는 $2B$**, 우리가 시각화하는 건 그 **절반인 $\omega_s = B$ 로 샘플링** 한 결과. 샘플링하면 주파수 스펙트럼이 주기 $B$ 로 *무한히* 반복된다:

$$X^s(j\omega) \;=\; \sum_{k=-\infty}^{\infty} X\bigl(j(\omega - kB)\bigr)$$

그러나 $X$ 의 support 가 $[-B, +B]$ 라서, $|k| \ge 2$ 인 복제본은 $[-B, +B]$ 바깥에 떨어진다. 신호의 본래 support 안에서 원본과 겹치는 alias 는 $k = \pm 1$ 두 개뿐:

- $\omega \in [0, +B]$: 원본 $X(\omega)$ + alias $X(\omega - B)$ (음의 절반이 양의 쪽으로 접혀 들어옴)
- $\omega \in [-B, 0]$: 원본 $X(\omega)$ + alias $X(\omega + B)$ (양의 절반이 음의 쪽으로 접혀 들어옴)

> **아이디어**: aliasing은 "지우는" 게 아니라 "**다른 aliasing으로 덮어서 상쇄**" 할 수 있다. 같은 정보를 담은 두 번째 신호 $y(t)$ 를 만들어 *alias 부분만 부호가 반대* 가 되게 하면, 두 결과를 더해 alias 가 완전히 소멸하고 원본만 남는다.

조건은 두 가지:

- $y(t)$ 도 **실수 신호** 여야 함 (아날로그 제약)
- $y(t)$ 의 spectrum $Y(j\omega)$ 가 alias 위치에서 $X$ 와 **부호 반대**

부호 반대를 만들려면 $Y$ 가 **기함수 같은 성질** 을 가져야 한다. 기함수면 $Y(\omega - B) = -Y(B - \omega)$ — alias 부호 반대.

In [15]:
# §5: 세 개의 분리된 figure 로 재구성
#   [5a] X (우함수) — 원본 + B만큼 평행이동한 복제본들 + 합 (alias 같은 부호로 덧셈)
#   [5b] Y (기함수, sgn(ω)·X) — 같은 구조, 단 alias 가 반대 부호로 부분 상쇄
#   [5c] alias 부호 비교 — observable band [-B,+B]만 잘라
#         alias_X (양) 와 alias_Y (음) 가 오른쪽 절반에서 ω축에 대해 거울 대칭임을 직접 시각화

sigma = 0.55
B = 3.0                                              # 대역폭, ω_s = B
omega_wide = np.linspace(-1.7*B, 1.7*B, 1501)         # [5a], [5b] — 양옆 ±2B 복제본까지 보이게
omega_band = np.linspace(-B, B, 1001)                 # [5c] — observable band 만

def X_even(w):
    g = sigma * np.sqrt(2*np.pi) * np.exp(-0.5 * (sigma*w)**2)
    return np.where(np.abs(w) <= B, g, 0.0)

def Y_odd(w):
    return np.sign(w) * X_even(w)

# k = -2, -1, 0, +1, +2 복제본
def replicas(fn, w):
    return [fn(w - k*B) for k in (-2, -1, 0, 1, 2)]

X_reps = replicas(X_even, omega_wide)
Y_reps = replicas(Y_odd,  omega_wide)
X_sum  = sum(X_reps)
Y_sum  = sum(Y_reps)


def _two_panel(reps, total, orig_color, alias_color, sig_label, title):
    """좌: 원본 + 복제본들, 우: 합. (5a/5b 공통 레이아웃)"""
    fig = make_subplots(
        rows=1, cols=2, horizontal_spacing=0.10,
        subplot_titles=(f"{sig_label} (ω) 원본 + B만큼 평행이동한 복제본들",
                        f"{sig_label}^s = Σ {sig_label}(ω−kB)  (실제 측정값)"),
    )

    comp = {
        -2: ("rgba(125,133,144,0.35)", "dot",  "k=−2"),
        -1: (alias_color,              "dash", f"alias  k=−1  ({sig_label}(ω+B))"),
         0: (orig_color,               None,   f"원본  k=0   ({sig_label}(ω))"),
        +1: (alias_color,              "dash", f"alias  k=+1  ({sig_label}(ω−B))"),
        +2: ("rgba(125,133,144,0.35)", "dot",  "k=+2"),
    }
    for k, rep in zip((-2,-1,0,1,2), reps):
        color, dash, label = comp[k]
        width = 4 if k == 0 else 2.5
        fig.add_trace(go.Scatter(x=omega_wide, y=rep, mode="lines",
                                  line=dict(color=color, width=width, dash=dash),
                                  name=label, showlegend=True),
                      row=1, col=1)
    # 합
    fig.add_trace(go.Scatter(x=omega_wide, y=total, mode="lines",
                              line=dict(color=_YELLOW, width=4),
                              name=f"{sig_label}^s  합", showlegend=False),
                  row=1, col=2)
    fig.add_trace(go.Scatter(x=omega_wide, y=reps[2], mode="lines",
                              line=dict(color=orig_color, width=1, dash="dot"),
                              name=f"{sig_label}(ω) 참고", showlegend=False),
                  row=1, col=2)

    for c in (1, 2):
        fig.add_vline(x=+B, line=dict(color=_RED, width=1, dash="dash"), row=1, col=c)
        fig.add_vline(x=-B, line=dict(color=_RED, width=1, dash="dash"), row=1, col=c)
        fig.add_vline(x=0,  line=dict(color=_AXIS, width=1, dash="dash"), row=1, col=c)
        fig.add_vrect(x0=-B, x1=+B, fillcolor="rgba(126,198,153,0.05)",
                      line_width=0, row=1, col=c)
        fig.update_xaxes(_2D_AXIS, title="ω  (붉은 점선 = 신호 support [−B, +B])",
                         row=1, col=c)
        fig.update_yaxes(_2D_AXIS, row=1, col=c)

    fig.update_layout(
        **_DEFAULT_LAYOUT, height=360,
        title=dict(text=title, x=0.02, xanchor="left", font=dict(color=_FG)),
        legend=dict(bgcolor="rgba(0,0,0,0)", font=dict(color=_FG, size=10),
                    x=1.02, y=0.95),
        showlegend=True,
    )
    for ann in fig["layout"]["annotations"]:
        ann["font"] = dict(color=_FG, size=12)
    return fig


# ── [5a] X (우함수 truncated Gaussian) ─────────────────────────────────
fig_a = _two_panel(
    X_reps, X_sum, orig_color=_GREEN, alias_color=_ORANGE, sig_label="X",
    title=f"[5a]  X — 우함수 truncated Gaussian.  "
          "alias 가 *같은 부호* 로 누적 → 합 진폭 두 배",
)
fig_a.show()

# ── [5b] Y (기함수, sgn(ω)·X) ───────────────────────────────────────────
fig_b = _two_panel(
    Y_reps, Y_sum, orig_color=_BLUE, alias_color=_RED, sig_label="Y",
    title=f"[5b]  Y = sgn(ω)·X — 기함수.  "
          "alias 가 *반대 부호* 로 작용 → 합이 부분 상쇄",
)
fig_b.show()

# ── [5c] alias 부호 비교 — observable band [-B, +B] 만 ─────────────────
# alias 전체 = k=±1 합 (이 범위에서 k≥2 복제본은 0). 양옆 ±2B 복제본 자른 모습.
alias_X = X_even(omega_band - B) + X_even(omega_band + B)
alias_Y = Y_odd(omega_band - B)  + Y_odd(omega_band + B)
y_amp = max(np.max(np.abs(alias_X)), np.max(np.abs(alias_Y))) * 1.15

fig_c = fig2d(
    title="[5c]  alias 부호 비교 (observable band [−B, +B] 만 잘라)  "
          "— 오른쪽 절반에서 두 곡선이 ω축에 대해 *거울 대칭*",
    height=420, xtitle="ω", ytitle="alias  값",
)
# alias_X — 양의 가우시안 (위)
fig_c.add_trace(go.Scatter(
    x=omega_band, y=alias_X, mode="lines",
    line=dict(color=_ORANGE, width=4),
    fill="tozeroy", fillcolor="rgba(255,166,87,0.28)",
    name="X-alias (우함수: ω축 대칭)",
))
# alias_Y — 기함수 (왼쪽 양, 오른쪽 음)
fig_c.add_trace(go.Scatter(
    x=omega_band, y=alias_Y, mode="lines",
    line=dict(color=_RED, width=4),
    fill="tozeroy", fillcolor="rgba(255,123,114,0.28)",
    name="Y-alias (기함수: 원점 대칭)",
))
# y=0, ω=0 강조선
fig_c.add_hline(y=0, line=dict(color=_AXIS, width=1.5))
fig_c.add_vline(x=0, line=dict(color=_AXIS, width=1, dash="dot"))
# observable band 경계
fig_c.add_vline(x=+B, line=dict(color=_RED, width=1, dash="dash"))
fig_c.add_vline(x=-B, line=dict(color=_RED, width=1, dash="dash"))
# 오른쪽 절반 = mirror 영역 음영
fig_c.add_vrect(x0=0, x1=+B, fillcolor="rgba(126,198,153,0.05)", line_width=0)

# 오른쪽 절반 거울 대칭을 강조하는 연결 점선들 — 샘플 ω 3개
for w_sample in (0.35*B, 0.55*B, 0.78*B):
    yx = np.interp(w_sample, omega_band, alias_X)
    yy = np.interp(w_sample, omega_band, alias_Y)
    fig_c.add_trace(go.Scatter(
        x=[w_sample, w_sample], y=[yx, yy], mode="lines",
        line=dict(color="rgba(230,237,243,0.45)", width=1.2, dash="dot"),
        showlegend=False, hoverinfo="skip",
    ))

# 영역 주석
fig_c.add_annotation(
    x=+0.55*B, y=+y_amp*0.78, text="<b>오른쪽 [0,+B]</b>:<br>alias_Y = −alias_X<br>→ 더하면 0",
    showarrow=False, font=dict(color=_FG, size=12),
    bgcolor="rgba(13,17,23,0.7)", borderwidth=0,
    xanchor="center",
)
fig_c.add_annotation(
    x=-0.55*B, y=+y_amp*0.78, text="<b>왼쪽 [−B,0]</b>:<br>alias_X = +alias_Y<br>(곡선 일치)",
    showarrow=False, font=dict(color=_FG, size=12),
    bgcolor="rgba(13,17,23,0.7)", borderwidth=0,
    xanchor="center",
)
fig_c.update_yaxes(range=[-y_amp, y_amp])
fig_c.update_xaxes(range=[-1.05*B, +1.05*B])
fig_c.update_layout(width=640, autosize=False)   # [5c] 가로 ~20% 축소
fig_c.show()

## [6] 기함수는 실수가 아니다 — Im 평면으로 옮기자

위에서 만든 $Y(j\omega)$는 *순실수 기함수*. 시간 영역에서 이건 **순허수 신호** ($x(t)$가 실수면 $X$ Re는 우, Im은 기 — 거꾸로 Re만 기함수이면 Hermitian 깨짐). 아날로그 불가.

해결: 기함수 정보를 $\omega$-Re 평면이 아니라 **$\omega$-Im 평면**에 올린다. $Y(j\omega)$를 **순허수이면서 기함수**가 되도록.

확인: 허수부가 기함수면 Hermitian 만족 → $y(t)$ 실수 신호 ✓. 그리고 alias 위치에서 부호 반대 ✓.

그러려면 $X(j\omega)$ (순실수 우함수)에서 $Y(j\omega)$ (순허수 기함수)를 얻는 변환이 필요하다:

- $\omega > 0$: 실수 $X$ → 음의 허수 → $H = -j$ 곱
- $\omega < 0$: 실수 $X$ → 양의 허수 → $H = +j$ 곱

즉

$$\boxed{H(j\omega) = -j\,\mathrm{sgn}(\omega)}$$

이게 **Hilbert transform**. 기하적으로는 $\omega$-Re 평면의 곡선을 $\omega$-Im 평면으로 90° 회전 (양/음 ω에서 부호 반대) — Re ↔ Im 평면 교환 + 부호 조정이다.

In [8]:
# 3D 시각화: X (Re 평면 우함수) → Y = H·X (Im 평면 기함수)
# 그리고 Hilbert H(jω) = -j·sgn(ω) 자체도 3D로.

sigma = 0.6
B = 4.0
omega = np.linspace(-B, B, 901)

X_real_even = sigma * np.sqrt(2*np.pi) * np.exp(-0.5 * (sigma*omega)**2)
H = -1j * np.sign(omega)                         # Hilbert 주파수 응답
Y = H * X_real_even                              # = -j·sgn(ω)·X — 순허수 기함수

# (a) X와 Y를 한 3D 씬에 같이
fig = fig3d(title='[6a] X(jω) (Re 평면 우함수)  →  Y = H·X (Im 평면 기함수)',
            height=560, xtitle='ω', ytitle='Re', ztitle='Im',
            aspect=(2.4, 1.0, 1.0), eye=(2.0, 1.6, 1.0))
add_central_axis(fig, (omega[0], omega[-1]))
# X는 Re 평면 위 곡선 (z=0)
fig.add_trace(go.Scatter3d(
    x=omega, y=X_real_even, z=np.zeros_like(omega), mode='lines',
    line=dict(color=_GREEN, width=5),
    name='X(jω)  ∈ Re-ω 평면 (우함수)'))
# Y는 Im 평면 위 곡선 (y=0, z=Im)
fig.add_trace(go.Scatter3d(
    x=omega, y=np.zeros_like(omega), z=Y.imag, mode='lines',
    line=dict(color=_PURPLE, width=5),
    name='Y(jω) = H·X  ∈ Im-ω 평면 (기함수)'))
# Re=0 평면 (Y가 들어가는 곳)을 살짝 강조
add_plane(fig, axis='y', value=0.0,
          u_range=(omega[0], omega[-1]),
          w_range=(float(np.min(Y.imag))*1.2, float(np.max(Y.imag))*1.2),
          color='rgba(191,145,243,0.07)', name='Re=0 평면')
fig.show()

# (b) Hilbert H(jω) 자체를 3D로 — 양의 ω에서 −j, 음의 ω에서 +j인 *단순한* 스텝
panel_H = SpectrumPlot(view='re_im', show_inset=True)
panel_H.show_spectrum(omega, H, flatten_noise=False)
panel_H.figure.layout.title.text = (
    '[6b] Hilbert  H(jω) = −j·sgn(ω)  — Re=0,  Im은 −1 / +1 스텝 (기함수)'
)
panel_H.figure.show()

## [7] 일반 실수 신호로 확장 — 전체 파이프라인을 3D로

[5]는 $X$ 가 *순실수 우함수*인 특수 경우. 같은 $H = -j\,\mathrm{sgn}(\omega)$ 가 *일반* 실수 신호 ($X_R$ 우함수 + $jX_I$ 기함수, support $[-B, +B]$)에서도 작동하는지 확인:

$$Y(j\omega) = -j\,\mathrm{sgn}(\omega)(X_R + jX_I) = -j\,\mathrm{sgn}(\omega)\,X_R \;+\; \mathrm{sgn}(\omega)\,X_I$$

- $X_R$ (우함수) × $-j\,\mathrm{sgn}(\omega)$ → 순허수 기함수
- $X_I$ (기함수) × $\mathrm{sgn}(\omega)$ → 순실수 우함수

**Re와 Im이 서로 교환되며 부호 조정.** $Y$ 도 Hermitian, $y(t)$ 실수 ✓

이제 두 신호를 각각 $\omega_s = B$ (half-Nyquist) 로 샘플링. observable band $[-B, +B]$ 안에서 $k = \pm 1$ alias 가 원본과 겹친다.

### 그림으로 본 상쇄

$\omega \in [0, +B]$ 에서 $X^s$ 와 $j Y^s$ 를 성분 별로 보면:

| 성분 | $X^s$ | $j Y^s = j\cdot(-j\,\mathrm{sgn}(\omega))\cdot X = \mathrm{sgn}(\omega)\,X$ | 관계 |
|---|---|---|---|
| 원본 $(k=0)$ | $X(\omega)$ | $+X(\omega)$ | **일치** → 더하면 $2X(\omega)$ |
| Alias $(k=+1)$ | $X(\omega-B)$ | $\mathrm{sgn}(\omega-B)\,X(\omega-B) = -X(\omega-B)$ | **원점 대칭** → 더하면 $0$ |

즉 두 채널을 같은 평면에 펼치면 *원본 곡선은 정확히 겹치고, alias 곡선은 원점을 기준으로 거울상* 이다. 더하면 원본은 두 배가 되고 alias 는 사라진다. 패널 ④가 이 기하를 직접 보여준다.

$$Z^s(\omega) \;=\; X^s(\omega) + j\,Y^s(\omega) \;=\; 2\,X(\omega) \quad (\omega \in [0, +B])$$

음의 절반 $X(\omega)$ on $[-B, 0]$ 은 Hermitian 대칭으로 디지털에서 $\overline{X(-\omega)}$ 로 재구성. 즉 **양의 절반만 측정해도 전체 신호 복원** — 절반 레이트로 동작하는 본질. 5단계 3D 패널로 직접 본다.

In [9]:
# 일반 실수 신호의 truncated 스펙트럼 X(jω) (Hermitian, support [-B, +B])
# ω_s = B 샘플링 → 원본 + alias component (k=±1만 [-B,+B] 안에 겹침)
# Hilbert 통과 → 변환된 원본·alias component
# 같은 이야기를 그림으로: ω∈[0,+B] 에서 X^s 와 j·Y^s 의 원본 성분은 일치, alias 성분은 원점 대칭
# 디지털 결합 X^s + j·Y^s → [0,+B]에서 2·X(ω) 복원

sigma = 0.55
B = 3.0
omega_full = np.linspace(-B, B, 1201)

def XR_fn(w):
    g = np.exp(-0.5 * (sigma*w)**2)
    return np.where(np.abs(w) <= B, g, 0.0)

def XI_fn(w):
    g = 0.7 * w * np.exp(-0.5 * (sigma*w)**2)
    return np.where(np.abs(w) <= B, g, 0.0)

def Xval(w):
    return XR_fn(w) + 1j * XI_fn(w)

def Yval(w):
    return -1j * np.sign(w) * Xval(w)

omega_band = np.linspace(-B, B, 1201)

X_orig    = Xval(omega_band)
X_alias_p = Xval(omega_band - B)
X_alias_m = Xval(omega_band + B)
X_sampled = X_orig + X_alias_p + X_alias_m

Y_orig    = Yval(omega_band)
Y_alias_p = Yval(omega_band - B)
Y_alias_m = Yval(omega_band + B)
Y_sampled = Y_orig + Y_alias_p + Y_alias_m

Z_sampled = X_sampled + 1j * Y_sampled

mask_inner = (omega_band > B/8) & (omega_band < 7*B/8)
expected_inner = 2 * Xval(omega_band[mask_inner])
err = float(np.max(np.abs(Z_sampled[mask_inner] - expected_inner))
            / np.max(np.abs(expected_inner)))
print(f'양의 절반 in-band (ω ∈ [B/8, 7B/8]) 에서  '
      f'max|Z^s − 2·X(ω)| / max|2X|  =  {err:.2e}')
print(f'→ 경계(sgn(0)·truncation) 외에는 alias 완전 상쇄 ✓')


def add_spectrum_trace(fig, ws, vals, *, color, name, width=4, dash=None):
    fig.add_trace(go.Scatter3d(
        x=ws, y=vals.real, z=vals.imag, mode='lines',
        line=dict(color=color, width=width, dash=dash), name=name,
    ))


# ── (1) 원본 X(jω) 3D ─────────────────────────────────────────────────
fig1 = fig3d(title='[7-①] 일반 실수 신호의 X(jω) (truncated, support [−B, +B])  '
                   '— Re 우함수 + Im 기함수 (Hermitian)',
             height=460, xtitle='ω', ytitle='Re X', ztitle='Im X',
             aspect=(2.4, 1.0, 1.0), eye=(2.1, 1.6, 1.0))
add_central_axis(fig1, (omega_full[0], omega_full[-1]))
add_spectrum_trace(fig1, omega_full, Xval(omega_full), color=_GREEN, name='X(jω) 원본')
fig1.show()

# ── (2) 샘플링 후 observable band — X^s 의 원본 + alias components ─────
fig2 = fig3d(title='[7-②] ω_s = B 샘플링 — observable band [−B,+B] 안 원본 + alias (k=±1)',
             height=460, xtitle='ω', ytitle='Re', ztitle='Im',
             aspect=(2.4, 1.0, 1.0), eye=(2.1, 1.6, 1.0))
add_central_axis(fig2, (-B, B))
add_spectrum_trace(fig2, omega_band, X_orig,    color=_GREEN,  name='X(ω) 원본')
add_spectrum_trace(fig2, omega_band, X_alias_p, color=_ORANGE, name='X(ω−B) alias k=+1 (강조)', dash='dash')
add_spectrum_trace(fig2, omega_band, X_alias_m, color=_ORANGE, name='X(ω+B) alias k=−1 (강조)', dash='dot')
add_spectrum_trace(fig2, omega_band, X_sampled, color=_YELLOW, name='X^s = 합 (실제 측정값)', width=2)
fig2.show()

# ── (3) Hilbert 통과 후 — Y^s 의 원본 + alias components ───────────────
fig3 = fig3d(title='[7-③] y(t) 채널 (H 통과) — alias component 가 *반대 부호* 로 회전',
             height=460, xtitle='ω', ytitle='Re', ztitle='Im',
             aspect=(2.4, 1.0, 1.0), eye=(2.1, 1.6, 1.0))
add_central_axis(fig3, (-B, B))
add_spectrum_trace(fig3, omega_band, Y_orig,    color=_BLUE, name='Y(ω) = −j·sgn(ω)·X(ω)')
add_spectrum_trace(fig3, omega_band, Y_alias_p, color=_RED,  name='Y(ω−B) alias k=+1 (반대 부호)', dash='dash')
add_spectrum_trace(fig3, omega_band, Y_alias_m, color=_RED,  name='Y(ω+B) alias k=−1 (반대 부호)', dash='dot')
add_spectrum_trace(fig3, omega_band, Y_sampled, color=_YELLOW, name='Y^s', width=2)
fig3.show()

# ── (4) 같은 이야기를 그림으로: X^s 와 j·Y^s 의 성분을 한 화면에 ─────────────
# ω ∈ [0, +B] 에서:  X^s 원본 = X(ω),   j·Y^s 원본 = +X(ω)   → 일치 (덧셈으로 두 배)
#                   X^s alias = X(ω−B), j·Y^s alias = −X(ω−B) → 원점 대칭 (더하면 0)
mask_pos = omega_band >= 0
op = omega_band[mask_pos]
X_orig_p_pos  = X_orig[mask_pos]
X_alias_p_pos = X_alias_p[mask_pos]
jY_orig_p_pos = 1j * Y_orig[mask_pos]       # = sgn(ω)·X(ω) = +X(ω) for ω>0
jY_alias_pos  = 1j * Y_alias_p[mask_pos]    # = sgn(ω−B)·X(ω−B) = −X(ω−B)

fig4 = fig3d(title='[7-④] 그림으로 본 상쇄 (ω∈[0,+B]):  '
                   'X^s 와 j·Y^s 의 원본 = 일치 (덧셈으로 ×2),  alias = 원점 대칭 (더하면 0)',
             height=520, xtitle='ω', ytitle='Re', ztitle='Im',
             aspect=(2.4, 1.0, 1.0), eye=(2.0, 1.7, 1.0))
add_central_axis(fig4, (0, B))
# 원본 성분 — 같음
add_spectrum_trace(fig4, op, X_orig_p_pos,  color=_GREEN,  name='X^s 원본 = X(ω)',                  width=6)
add_spectrum_trace(fig4, op, jY_orig_p_pos, color=_WHITE,  name='j·Y^s 원본 = +X(ω)  ★ 같음',        width=2, dash='dot')
# alias 성분 — 원점 대칭
add_spectrum_trace(fig4, op, X_alias_p_pos, color=_ORANGE, name='X^s alias = X(ω−B)',               width=6)
add_spectrum_trace(fig4, op, jY_alias_pos,  color=_RED,    name='j·Y^s alias = −X(ω−B)  ★ 원점 대칭', width=6)
# 두 alias 곡선을 같은 ω 위치에서 잇는 연결선 (원점 대칭 시각화) — 일부 샘플만
sample_idx = np.linspace(20, len(op)-20, 9, dtype=int)
for k in sample_idx:
    fig4.add_trace(go.Scatter3d(
        x=[op[k], op[k], op[k]],
        y=[X_alias_p_pos[k].real, 0, jY_alias_pos[k].real],
        z=[X_alias_p_pos[k].imag, 0, jY_alias_pos[k].imag],
        mode='lines',
        line=dict(color='rgba(255,123,114,0.45)', width=2, dash='dot'),
        showlegend=False, hoverinfo='skip',
    ))
fig4.show()

# ── (5) 디지털 결합 Z^s = X^s + j·Y^s ──────────────────────────────────
# 위의 기하적 사실 그대로:  원본 끼리 더해 2X(ω),  alias 끼리 더해 0
fig5 = fig3d(title='[7-⑤] 디지털 결합 Z^s = X^s + j·Y^s  '
                   '— [0,+B] 에서 2·X(ω) 복원 (alias 상쇄)',
             height=460, xtitle='ω', ytitle='Re', ztitle='Im',
             aspect=(2.4, 1.0, 1.0), eye=(2.1, 1.6, 1.0))
add_central_axis(fig5, (-B, B))
mask_pos2 = omega_band >= 0
add_spectrum_trace(fig5, omega_band[mask_pos2], (2*Xval(omega_band[mask_pos2])),
                    color=_AXIS, name='기대치 2·X(ω) on [0,+B]', width=2, dash='dot')
add_spectrum_trace(fig5, omega_band, Z_sampled, color=_GREEN, name='실제 합  Z^s', width=5)
fig5.show()

양의 절반 in-band (ω ∈ [B/8, 7B/8]) 에서  max|Z^s − 2·X(ω)| / max|2X|  =  1.49e-16
→ 경계(sgn(0)·truncation) 외에는 alias 완전 상쇄 ✓


## [8] 시스템 다이어그램 + 비인과 임펄스 응답 $h(t) = 1/(\pi t)$

전체 시스템:

```
                      ┌──── ADC (ω_s = B) ─────► x[n] ──────┐
                      │                                      │
   x(t) ──────────────┤                                      ├──► [DSP] ──► 2X(jω)
                      │                                      │
                      └──── H(jω)=−j·sgn(ω) ─── ADC ──► y[n] ──► (+j)·Y(jω) ──┘
                            (아날로그)         (ω_s = B)      (디지털)
```

한 번의 아날로그 변환(Hilbert), 두 번의 절반 레이트 샘플링, 디지털 곱셈 + 더하기.

### $h(t)$ 구하기

$H(j\omega) = -j\,\mathrm{sgn}(\omega)$의 역변환 (수렴 인자 $e^{-\epsilon|\omega|}$ 도입):

$$h(t) = \lim_{\epsilon \to 0^+} \frac{-j}{2\pi}\!\left[\frac{1}{\epsilon - jt} - \frac{1}{\epsilon + jt}\right] \;=\; \boxed{\dfrac{1}{\pi t}}$$

표준 Hilbert transform 임펄스 응답. $t < 0$에서도 0이 아닌 **비인과적** 시스템 — 출력이 미래 입력에 의존. 정확한 $H$는 실시간 아날로그 회로로 *구현 불가능*. 회로 수준에서 어떻게 우회하는지가 다음 절의 주제다.

In [10]:
# h(t) = 1/(πt) — 양/음 t 두 구간을 따로 생성. t=0 자체를 배열에서 빼므로
# 0-나눗셈 자체가 일어나지 않는다 (no masking, no warning).
t_neg = np.linspace(-3.0, -1e-3, 1000)
t_pos = np.linspace(+1e-3, +3.0, 1000)
h_neg = 1.0 / (np.pi * t_neg)
h_pos = 1.0 / (np.pi * t_pos)

fig = fig2d(title='h(t) = 1/(π t)  — Hilbert 임펄스 응답.  t<0에서도 0이 아님 → *비인과*',
            height=300, xtitle='t', ytitle='h(t)')
fig.add_trace(go.Scatter(x=t_neg, y=h_neg, mode='lines',
                          line=dict(color=_RED, width=3),
                          name='t < 0  (미래 입력 의존 — 구현 불가)'))
fig.add_trace(go.Scatter(x=t_pos, y=h_pos, mode='lines',
                          line=dict(color=_GREEN, width=3),
                          name='t > 0  (인과적 부분)'))
fig.add_vline(x=0, line=dict(color=_AXIS, width=1, dash='dash'))
fig.update_yaxes(range=[-4, 4])
fig.show()

## [9] 회로 구현의 본질적 제약 — 1차 active all-pass

전자회로 관점에서 **magnitude 1을 유지하는 LTI 회로는 RC all-pass cascade가 사실상 유일**하다. 표준 1차 active all-pass:

![1차 active all-pass 단](../data/allpass_circuit.svg)

전달함수 (가상 단락 + $R_2$ 두 개 매칭):

$$H(s) = \dfrac{1 - sR_1C}{1 + sR_1C}, \qquad |H(j\omega)| = 1, \qquad \phi(\omega) = -2\arctan(\omega R_1 C)$$

- **Pole**: $s = -1/(R_1C)$ (LHP, 안정)
- **Zero**: $s = +1/(R_1C)$ (RHP, all-pass 특성 — pole의 거울상)
- $R_1, C$가 pole 위치 $p = 1/(R_1C)$ 결정, $R_2$는 매칭만 중요

## [10] All-pass의 한계 — Monotonic Phase

1차 all-pass의 phase $\phi(\omega) = -2\arctan(\omega/p)$는 $\omega = 0$에서 0, $\omega \to \infty$에서 $-\pi$로 **단조 감소**.

$N$단 cascade도 마찬가지: 각 단의 phase가 단조 감소 함수의 합이라 전체도 단조. 일반적으로 all-pass의 group delay

$$\tau_g(\omega) = \sum_k \frac{2 p_k}{p_k^2 + \omega^2} > 0$$

모든 항이 양수 → phase는 항상 단조 감소. **$\pm\pi/2$ plateau 직접 구현 불가능.**

따라서 단일 all-pass cascade로는 Hilbert를 만들 수 없음.

In [11]:
# 1차 all-pass와 3단 cascade의 phase response 비교
omega = np.logspace(-2, 2, 1001)

def allpass_phase(omega, poles):
    """N개 pole의 1차 all-pass cascade phase."""
    return -2 * np.sum([np.arctan(omega/p) for p in poles], axis=0)

phi1 = allpass_phase(omega, [1.0])
phi3 = allpass_phase(omega, [0.3, 1.0, 3.0])

fig = fig2d(title='[10] 1차 all-pass와 cascade의 phase — 단조 감소, plateau 불가',
            height=320, xtitle='ω  (log scale)', ytitle='arg H  (rad)')
fig.add_trace(go.Scatter(x=omega, y=phi1, mode='lines',
                          line=dict(color=_GREEN, width=3),
                          name='1차 all-pass (p=1)'))
fig.add_trace(go.Scatter(x=omega, y=phi3, mode='lines',
                          line=dict(color=_ORANGE, width=3),
                          name='3단 cascade (p=0.3, 1, 3)'))
fig.add_hline(y=-np.pi/2, line=dict(color=_RED, width=1, dash='dash'),
              annotation_text='목표  −π/2',
              annotation=dict(font=dict(color=_RED)))
fig.add_hline(y=-np.pi, line=dict(color=_AXIS, width=1, dash='dot'))
fig.update_xaxes(type='log')
fig.show()

## [11] 우회: 두 채널의 *차이* 로 Plateau 만들기

각 채널은 monotonic이지만, 두 채널 사이 phase **차이**는 plateau 가능. Pole 위치만 어긋나게 배치하면 in-band에서 차이가 상수 $-\pi/2$ 근처 유지.

채널 I와 Q에 서로 다른 cutoff 조합:

$$\phi_I(\omega) = -2\sum_k \arctan(\omega/p_{I,k}), \qquad \phi_Q(\omega) = -2\sum_k \arctan(\omega/p_{Q,k})$$

$\{p_{I,k}\}, \{p_{Q,k}\}$를 적절히 설계해 in-band에서 $\phi_Q(\omega) - \phi_I(\omega) \approx -\pi/2$.

아래 5단 polyphase에서 직접 보자: 두 채널 모두 $|H|=1$, phase는 단조 감소, 두 phase의 *차이*가 in-band에서 평탄한 $-90°$ 라인을 그린다.

Stage 수 $N$ 증가에 따라 in-band 위상 오차가 **기하급수적**으로 감소 — image rejection 표를 함께 본다.

$\mathrm{IR} \approx -20\log_{10}|\tan(\Delta\phi_{\text{err}}/2)|$

In [12]:
# 짝쌍 polyphase: 각 단마다 중심 ω_k에서  p_I[k] = ω_k·√R, p_Q[k] = ω_k/√R.
# R은 각 N에 맞게 bisection 으로 자동 튜닝 — 중심 ω_c 에서 Δφ = −π/2 가 정확히 떨어지도록.
# 더 많은 단(N↑) 일수록 plateau가 더 평탄.

def find_R(N, omega_c=1.0, band_ratio=100.0, target=-np.pi/2):
    """중심 ω_c 에서 Δφ = target 이 되도록 R을 bisection 으로 찾는다."""
    rho = band_ratio ** (1.0 / max(N, 1))
    centers = omega_c * rho ** (np.arange(N) - (N-1)/2.0)
    def dphi_at_center(R):
        sqrt_R = np.sqrt(R)
        p_I = centers * sqrt_R
        p_Q = centers / sqrt_R
        return (allpass_phase(np.array([omega_c]), p_Q)
                - allpass_phase(np.array([omega_c]), p_I))[0]
    lo, hi = 1.001, 20.0
    for _ in range(60):
        mid = 0.5*(lo + hi)
        if dphi_at_center(mid) > target:    # 너무 작음 (덜 음수) → R↑
            lo = mid
        else:
            hi = mid
    return 0.5*(lo + hi)

def polyphase_poles(N, omega_c=1.0, band_ratio=100.0):
    """짝쌍 polyphase pole. R은 자동 튜닝되어 중심에서 Δφ = −π/2."""
    R = find_R(N, omega_c, band_ratio)
    rho = band_ratio ** (1.0 / max(N, 1))
    centers = omega_c * rho ** (np.arange(N) - (N-1)/2.0)
    sqrt_R = np.sqrt(R)
    return centers * sqrt_R, centers / sqrt_R


# 5단 vs 8단 비교: 2행 × 3열 (|H|, arg H, Δφ)
configs = [(5, 100.0), (8, 100.0)]   # 같은 band ratio로 N↑ 효과만 비교

fig = make_subplots(rows=2, cols=3, horizontal_spacing=0.08, vertical_spacing=0.18,
                    subplot_titles=(
                        "5단  |H| — 두 채널 모두 1",
                        "5단  arg H — 둘 다 단조 감소",
                        "5단  Δφ — in-band 중심에 −π/2 plateau",
                        "8단  |H|",
                        "8단  arg H",
                        "8단  Δφ — 단 수 ↑ → plateau 가 더 넓고 평탄",
                    ))

omega = np.logspace(-3, 3, 2401)

for row, (N, br) in enumerate(configs, start=1):
    p_I, p_Q = polyphase_poles(N, omega_c=1.0, band_ratio=br)
    phi_I = allpass_phase(omega, p_I)
    phi_Q = allpass_phase(omega, p_Q)
    mag_I = np.ones_like(omega)
    mag_Q = np.ones_like(omega)
    dphi = phi_Q - phi_I

    in_band = (omega >= 1/np.sqrt(br)) & (omega <= np.sqrt(br))

    fig.add_trace(go.Scatter(x=omega, y=mag_I, mode="lines",
                              line=dict(color=_GREEN, width=3), showlegend=False),
                  row=row, col=1)
    fig.add_trace(go.Scatter(x=omega, y=mag_Q, mode="lines",
                              line=dict(color=_ORANGE, width=3, dash="dash"), showlegend=False),
                  row=row, col=1)
    fig.add_trace(go.Scatter(x=omega, y=phi_I, mode="lines",
                              line=dict(color=_GREEN, width=3), showlegend=False),
                  row=row, col=2)
    fig.add_trace(go.Scatter(x=omega, y=phi_Q, mode="lines",
                              line=dict(color=_ORANGE, width=3), showlegend=False),
                  row=row, col=2)
    fig.add_trace(go.Scatter(x=omega, y=dphi, mode="lines",
                              line=dict(color=_BLUE, width=3), showlegend=False),
                  row=row, col=3)
    fig.add_hline(y=-np.pi/2, line=dict(color=_RED, width=1, dash="dash"),
                  row=row, col=3)
    fig.add_vrect(x0=float(omega[in_band].min()), x1=float(omega[in_band].max()),
                  fillcolor="rgba(126,198,153,0.07)", line_width=0,
                  row=row, col=3)

    for c in (1, 2, 3):
        fig.update_xaxes(_2D_AXIS, type="log", title="ω", row=row, col=c)
        fig.update_yaxes(_2D_AXIS, row=row, col=c)
    fig.update_yaxes(range=[0.95, 1.05], row=row, col=1)
    fig.update_yaxes(range=[-np.pi/2 - 1.2, -np.pi/2 + 1.2], row=row, col=3)

fig.update_layout(
    **_DEFAULT_LAYOUT, height=620,
    title=dict(text="[11] 5단 vs 8단 paired polyphase 비교  "
                    "— 단 수가 늘수록 −π/2 plateau 가 더 넓고 평탄해진다",
               x=0.02, xanchor="left", font=dict(color=_FG)),
    showlegend=False,
)
for ann in fig["layout"]["annotations"]:
    ann["font"] = dict(color=_FG, size=11)
fig.show()


# Image rejection 표 — 5단 vs 8단 비교에 집중
print(f'\n{"Stages":>7}  {"Band ratio":>11}  {"Tuned R":>9}  '
      f'{"Δφ max err (deg)":>17}  {"Image rejection (dB)":>20}')
print("-" * 78)
for N_try, br_try in configs:
    p_I_t, p_Q_t = polyphase_poles(N_try, omega_c=1.0, band_ratio=br_try)
    R_t = find_R(N_try, 1.0, br_try)
    margin = 0.3      # plateau 중심부만 평가 (in-band 가장자리 제외)
    wlo = 1/np.sqrt(br_try)/margin
    whi = np.sqrt(br_try)*margin
    w_eval = np.logspace(np.log10(wlo), np.log10(whi), 2001)
    err = (allpass_phase(w_eval, p_Q_t) - allpass_phase(w_eval, p_I_t)) - (-np.pi/2)
    e_max = float(np.max(np.abs(err)))
    IR = -20*np.log10(np.tan(e_max/2)) if e_max > 1e-12 else float("inf")
    print(f"{N_try:>7}  {br_try:>11g}  {R_t:>9.3f}  "
          f"{np.degrees(e_max):>17.3f}  {IR:>20.1f}")
print()
print("NB) 단순 짝쌍 설계는 N을 늘려도 ripple 진폭이 거의 같다 (oscillation 주기만 빨라짐).")
print("    그래프에서는 8단의 wiggle 이 5단보다 잦은 게 보이지만, 평균 ripple 은 비슷.")
print("    Saraga elliptic 설계라야 N↑ 시 ripple 이 기하급수적 감소 → 35–75 dB 가능.")


 Stages   Band ratio    Tuned R   Δφ max err (deg)  Image rejection (dB)
------------------------------------------------------------------------------
      5          100      1.692              8.091                  23.0
      8          100      1.390              8.178                  22.9

NB) 단순 짝쌍 설계는 N을 늘려도 ripple 진폭이 거의 같다 (oscillation 주기만 빨라짐).
    그래프에서는 8단의 wiggle 이 5단보다 잦은 게 보이지만, 평균 ripple 은 비슷.
    Saraga elliptic 설계라야 N↑ 시 ripple 이 기하급수적 감소 → 35–75 dB 가능.


## [12]–[13] 공통 phase $\alpha(\omega)$ — 부산물이지만 aliasing 제거에는 무해

두 채널 응답을 다시 쓰면:

$$H_I(j\omega) = e^{j\alpha(\omega)}, \qquad H_Q(j\omega) = -j\,\mathrm{sgn}(\omega) \cdot e^{j\alpha(\omega)}$$

여기서 $\alpha(\omega) = \phi_I(\omega)$ — monotonic 감소 함수, **0이 아닌 어떤 phase가 어쩔 수 없이 따라옴**. 이게 polyphase의 본질적 부산물.

### α가 있어도 Aliasing 제거 가능

$\omega_s = B$ 샘플링 후 ($\omega > 0$, alias는 $\omega-B < 0$). $A := e^{j\alpha(\omega)}X(\omega)$, $B := e^{j\alpha(\omega-B)}X(\omega-B)$ 두면:

$$Y_I^s = A + B, \qquad Y_Q^s = -jA + jB$$

**디지털 결합** $Y_I^s + jY_Q^s$:

$$= (A + B) + j(-jA + jB) = (A + B) + (A - B) = 2A = 2e^{j\alpha(\omega)}X(\omega)$$

Alias 완전 상쇄. 디지털에서 $e^{-j\alpha(\omega)}$ 곱해 $\alpha$ 보정 → $X(\omega)$ 복원.

**짚어둘 점**: $\alpha(\omega)$가 임의 함수여도 두 채널이 같은 $\alpha$를 공유하면 alias 상쇄 작동. 디지털 후처리에서 흡수 가능. 특히 $\alpha(\omega) = -\omega\tau_0$ (linear phase)면 $e^{-j\alpha}$ 보정이 단순 시간 지연 $-\tau_0$로 환원되어 후처리가 trivial.

In [13]:
# 짝쌍 polyphase로 α(ω) 시각화 + 수치 시뮬레이션:
#   truncated 실수 신호 → polyphase 두 채널 → ω_s=B 샘플링 → 결합 → α 보정 → X 복원

N = 5
band_ratio = 100.0
p_I, p_Q = polyphase_poles(N, omega_c=1.0, band_ratio=band_ratio)

omega_view = np.logspace(-1.5, 1.5, 1001)
alpha = allpass_phase(omega_view, p_I)

fig_a = fig2d(title="[12] 공통 phase  α(ω) = arg H_I(jω)  — monotonic, 0이 아닌 부산물",
              height=300, xtitle="ω (log)", ytitle="α(ω)  rad")
fig_a.add_trace(go.Scatter(x=omega_view, y=alpha, mode="lines",
                            line=dict(color=_BLUE, width=3), name="α(ω)"))
fig_a.update_xaxes(type="log")
fig_a.show()

# 수치 검증: truncated 실수 신호 + polyphase + ω_s=B 샘플링 + 결합 + α 보정
sigma = 0.55
B = 3.0
omega_full = np.linspace(-B, B, 2001)
def Xval3(w):
    g_R = np.exp(-0.5 * (sigma*w)**2)
    g_I = 0.7 * w * np.exp(-0.5 * (sigma*w)**2)
    mask = np.abs(w) <= B
    return np.where(mask, g_R, 0.0) + 1j * np.where(mask, g_I, 0.0)

def polyphase_H(omega, poles):
    """1차 all-pass cascade 복소 응답."""
    out = np.ones_like(omega, dtype=complex)
    for p in poles:
        out = out * (1 - 1j*omega/p) / (1 + 1j*omega/p)
    return out

# observable band [-B, +B] — k=±1 alias만 [-B,+B] 안에서 원본과 겹침
omega_band = np.linspace(-B + 1e-3, B - 1e-3, 1001)
H_I_0  = polyphase_H(omega_band,     p_I)
H_Q_0  = polyphase_H(omega_band,     p_Q)
H_I_p  = polyphase_H(omega_band - B, p_I)
H_Q_p  = polyphase_H(omega_band - B, p_Q)
H_I_m  = polyphase_H(omega_band + B, p_I)
H_Q_m  = polyphase_H(omega_band + B, p_Q)

Y_I_s = H_I_0*Xval3(omega_band) + H_I_p*Xval3(omega_band - B) + H_I_m*Xval3(omega_band + B)
Y_Q_s = H_Q_0*Xval3(omega_band) + H_Q_p*Xval3(omega_band - B) + H_Q_m*Xval3(omega_band + B)

combined = Y_I_s + 1j * Y_Q_s
# α(ω) 보정
corrected = combined * np.conj(H_I_0)

# 비교: in-band 안쪽 (ω ∈ [B/8, 7B/8]) 에서 2·X(ω) 복원 확인
mask_inner = (omega_band > B/8) & (omega_band < 7*B/8)
expected_inner = 2 * Xval3(omega_band[mask_inner])
err = float(np.max(np.abs(corrected[mask_inner] - expected_inner))
            / np.max(np.abs(expected_inner)))

print(f"5단 paired polyphase, ω_s = B = {B}")
print(f"  ω ∈ [B/8, 7B/8] 에서  max|corrected − 2·X(ω)| / max|2X|  =  {err:.2e}")
print(f"  → α 보정 + alias 상쇄 후 양의 절반 복원. ")
print(f"     (잔여 오차는 polyphase 위상 plateau 의 ripple — Saraga 설계로 개선 가능)")

fig_b = fig3d(title="[13] polyphase + α 보정 후 — 양의 절반 [0,+B] 에서 2·X(ω) 복원",
              height=460, xtitle="ω", ytitle="Re", ztitle="Im",
              aspect=(2.4, 1.0, 1.0), eye=(2.0, 1.6, 1.0))
add_central_axis(fig_b, (-B, B))
mask_pos = omega_band > 0
add_spectrum_trace(fig_b, omega_band[mask_pos], 2*Xval3(omega_band[mask_pos]),
                    color=_AXIS, name="기대치 2·X(ω) on [0,+B]", width=2, dash="dot")
add_spectrum_trace(fig_b, omega_band, corrected,
                    color=_GREEN, name="polyphase + α 보정 결과", width=5)
fig_b.show()

5단 paired polyphase, ω_s = B = 3.0
  ω ∈ [B/8, 7B/8] 에서  max|corrected − 2·X(ω)| / max|2X|  =  8.96e-02
  → α 보정 + alias 상쇄 후 양의 절반 복원. 
     (잔여 오차는 polyphase 위상 plateau 의 ripple — Saraga 설계로 개선 가능)


## [14] Polyphase Allpass Network — 전체 구조

각 채널 $N$단 1차 RC all-pass cascade, 두 채널 병렬:

```
              I 채널
        ┌─ [단 1] → [단 2] → ... → [단 N] ──► y_I
   x(t)─┤    p_I,1    p_I,2          p_I,N
        │
        └─ [단 1] → [단 2] → ... → [단 N] ──► y_Q
              Q 채널
             p_Q,1    p_Q,2          p_Q,N
```

각 단은 [9]의 RC + op-amp 1차 all-pass. Pole 위치 기하급수적 분포 (Saraga 표준 설계):

$$p_{I,k} = \omega_c \cdot \rho_k / \sqrt{a}, \qquad p_{Q,k} = \omega_c \cdot \rho_k \cdot \sqrt{a}$$

$\omega_c$: 대역 기하 중심, $\rho_k = (\omega_H/\omega_L)^{(2k-1-N)/(2N)}$, $a \approx 1.5$.

같은 die 위 RC 매칭이 우수하여 부품 절대 tolerance가 아닌 **상대 비율**이 성능 좌우. 5–10단으로 image rejection 35–75 dB 달성 (위 [11] 표).

## [15] 응용 — 이름들

이 방법은 통신 시스템에서 다음 이름들로 알려져 있다:

- **Phasing method** — SSB (single-sideband) 변조의 표준 기법 중 하나
- **Hartley modulator** — 90° phase splitter 기반 모듈레이터
- **IQ sampling / quadrature sampling** — 현대 SDR(software-defined radio)의 기본 구조
- **Analytic signal sampling** — 대역제한 신호를 절반 레이트로 받는 일반적 명칭

활용 분야:

- 무선 통신 수신기 (SSB, FM, 디지털 변조)
- 레이더 / 소나 신호 처리
- Quadrature ADC (RF front-end)
- 의료 영상 (MRI 신호 처리 — 복소 k-space, ultrasound beamforming)
- Direction-of-arrival (DOA) estimation

표준 polyphase 설계는 Saraga(1942), Bedrosian(1980) 등에서 정립되었고, 현대 RF / SDR 칩에 광범위하게 통합되어 있다.

---

### 한 줄 요약

**$x(t)$가 실수라는 *유일한* 가정만으로 샘플레이트를 절반으로 줄일 수 있다.** 그 가정이 Hermitian 대칭을 강제하고, 두 번째 Hilbert-변환된 채널과의 결합으로 aliasing이 정확히 상쇄되기 때문. 비인과적인 이상 Hilbert는 polyphase allpass network로 *공통 phase $\alpha(\omega)$* 라는 부산물을 감수하고 근사할 수 있고, 그 부산물도 디지털 후처리에서 흡수된다.